# 01 — Data Cleaning

**⚠️ All data used here is SYNTHETIC / SIMULATED.** See `../data/DATA_DICTIONARY.md` for full provenance. This notebook does not touch any real Sanofi data.

## Business question
Before any analysis can be trusted, is the raw data usable? Specifically:
- Are there duplicate records that would double-count sales?
- Are there inconsistent labels that would silently split one region into two in a GROUP BY?
- Are there data-entry errors that would distort averages?
- Does the recall/relaunch timeline behave the way the public record says it should?

This notebook answers those questions, fixes what can be fixed, and produces the cleaned tables that every later notebook builds on. Every issue found here was **deliberately injected** into the synthetic data (see `DATA_DICTIONARY.md` §4) so this notebook has something real to do — in a real engagement this step is where you'd find the equivalent issues in an actual company's systems.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

DATA = '../data/'
sales    = pd.read_csv(DATA + 'sales_data.csv', parse_dates=['date'])
hcp      = pd.read_csv(DATA + 'hcp_data.csv')
consumer = pd.read_csv(DATA + 'consumer_data.csv')
competitor = pd.read_csv(DATA + 'competitor_data.csv')
campaign = pd.read_csv(DATA + 'campaign_data.csv', parse_dates=['month'])
territory = pd.read_csv(DATA + 'territory_data.csv')

print(f"sales:      {sales.shape}")
print(f"hcp:        {hcp.shape}")
print(f"consumer:   {consumer.shape}")
print(f"competitor: {competitor.shape}")
print(f"campaign:   {campaign.shape}")
print(f"territory:  {territory.shape}")


sales:      (12735, 13)
hcp:        (2500, 14)
consumer:   (3000, 15)
competitor: (8, 17)
campaign:   (120, 14)
territory:  (36, 14)


## 1. Sales data — duplicates

In [2]:
key_cols = ['date', 'territory_id', 'channel', 'product']
n_dupes = sales.duplicated(subset=key_cols).sum()
print(f"Exact-duplicate rows on {key_cols}: {n_dupes}")

sales_clean = sales.drop_duplicates(subset=key_cols, keep='first').copy()
print(f"Rows before: {len(sales):,}  |  after de-dup: {len(sales_clean):,}")


Exact-duplicate rows on ['date', 'territory_id', 'channel', 'product']: 63
Rows before: 12,735  |  after de-dup: 12,672


## 2. Sales data — inconsistent state labels

A `GROUP BY state` before fixing this would silently create duplicate rows for the same state (e.g. "Delhi" and "NCT of Delhi" counted separately).

In [3]:
print("Distinct state labels (raw):", sorted(sales_clean.state.unique()))

state_map = {
    'NCT of Delhi': 'Delhi',
    'Tamil nadu': 'Tamil Nadu',
    'UP': 'Uttar Pradesh',
    'maharashtra': 'Maharashtra',
}
n_affected = sales_clean.state.isin(state_map).sum()
sales_clean['state'] = sales_clean.state.replace(state_map)
print(f"\nRows relabelled: {n_affected}")
print("Distinct state labels (clean):", sales_clean.state.nunique(), "-> matches 25 known states/UTs in the territory list? ",
      set(sales_clean.state) == set(territory.state))


Distinct state labels (raw): ['Andhra Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Delhi', 'Gujarat', 'Haryana', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'NCT of Delhi', 'Odisha', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Tamil nadu', 'Telangana', 'UP', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'maharashtra']

Rows relabelled: 47
Distinct state labels (clean): 21 -> matches 25 known states/UTs in the territory list?  True


## 3. Sales data — the recall window

Public record: DePURA Kids was recalled from a March 2024 letter, product returns would show up around April 2024, the product was off shelves through mid-2025, and was back on shelves by September 2025 (`research/sources.md` [S3][S5]). Confirming the data actually reflects this before trusting anything built on top of it.

In [4]:
phase_counts = (
    sales_clean.assign(
        market_phase=np.select(
            [
                sales_clean.date < '2024-03-01',
                sales_clean.date == '2024-03-01',
                sales_clean.date == '2024-04-01',
                (sales_clean.date > '2024-04-01') & (sales_clean.date < '2025-09-01'),
            ],
            ['pre_recall', 'recall_partial', 'recall_returns', 'off_market'],
            default='post_relaunch',
        )
    )
    .groupby('market_phase')
    .agg(rows=('units_sold', 'size'), total_units=('units_sold', 'sum'), total_revenue=('revenue', 'sum'))
)
phase_counts


,rows,total_units,total_revenue
market_phase,,,
off_market,4608,0,0.00
post_relaunch,3456,427488,49329548.46
pre_recall,4032,743935,87373780.26
recall_partial,288,15716,1845225.43
recall_returns,288,-13486,-1685453.00


In [5]:
# Sanity checks against the public timeline
off_market = sales_clean[(sales_clean.date > '2024-04-01') & (sales_clean.date < '2025-09-01')]
print("Off-market months should have zero units:", (off_market.units_sold == 0).all())

returns_month = sales_clean[sales_clean.date == '2024-04-01']
print("April 2024 (returns) has negative units in", (returns_month.units_sold < 0).sum(), "of", len(returns_month), "rows")

relaunch_month = sales_clean[sales_clean.date == '2025-09-01']
print("September 2025 (relaunch) total units:", relaunch_month.units_sold.sum(), "(should be > 0)")


Off-market months should have zero units: True
April 2024 (returns) has negative units in 144 of 288 rows
September 2025 (relaunch) total units: 19432 (should be > 0)


**Reading the recall/returns rows.** The negative-unit rows in April 2024 represent recalled stock being returned from trade — a real accounting event, not a data error. They should stay in the data (dropping them would understate what happened) but must be **excluded from any "average monthly demand" calculation**, or they will drag pre-recall averages down for the wrong reason. We tag `market_phase` on the clean table so every later notebook can filter deliberately rather than by accident.

In [6]:
sales_clean['market_phase'] = np.select(
    [
        sales_clean.date < '2024-03-01',
        sales_clean.date == '2024-03-01',
        sales_clean.date == '2024-04-01',
        (sales_clean.date > '2024-04-01') & (sales_clean.date < '2025-09-01'),
    ],
    ['pre_recall', 'recall_partial', 'recall_returns', 'off_market'],
    default='post_relaunch',
)

# "Active demand" view used for trend/average calculations throughout the
# project: excludes off-market zero rows and the one-off returns month.
sales_active = sales_clean[
    (sales_clean.market_phase.isin(['pre_recall', 'recall_partial', 'post_relaunch']))
    & (sales_clean.units_sold > 0)
].copy()

print(f"sales_clean:  {len(sales_clean):,} rows (kept, phase-tagged)")
print(f"sales_active: {len(sales_active):,} rows (used for demand trend/average calculations)")


sales_clean:  12,672 rows (kept, phase-tagged)
sales_active: 7,776 rows (used for demand trend/average calculations)


## 4. Sales data — missing discount values

`discount_pct` is legitimately null for off-market rows (no sale, no discount). It is also null for ~1% of *active* rows, which looks like a data-capture gap rather than a real absence of any discount. We impute those with the channel-month median, which is a defensible approach in a real trade-reporting pipeline where a discount field can go missing without the sale itself being unrecorded.

In [7]:
active_mask = sales_clean.market_phase.isin(['pre_recall', 'recall_partial', 'post_relaunch']) & (sales_clean.units_sold > 0)
n_null_active = sales_clean.loc[active_mask, 'discount_pct'].isna().sum()
print(f"Null discount_pct among active-demand rows: {n_null_active} of {active_mask.sum()}")

# Impute with channel + market_phase median (off-market rows are left null on purpose)
medians = sales_clean.loc[active_mask].groupby(['channel', 'market_phase'])['discount_pct'].transform('median')
sales_clean.loc[active_mask, 'discount_pct'] = sales_clean.loc[active_mask, 'discount_pct'].fillna(medians)
sales_active['discount_pct'] = sales_active['discount_pct'].fillna(
    sales_active.groupby(['channel', 'market_phase'])['discount_pct'].transform('median')
)

print("Remaining nulls in active rows after imputation:",
      sales_clean.loc[active_mask, 'discount_pct'].isna().sum())


Null discount_pct among active-demand rows: 126 of 7776
Remaining nulls in active rows after imputation: 0


## 5. HCP data — entry errors

In [8]:
print("monthly_patient_volume summary (raw):")
print(hcp.monthly_patient_volume.describe())

n_errors = (hcp.monthly_patient_volume == 9999).sum()
print(f"\nRows with the placeholder/entry-error value 9999: {n_errors}")
hcp.loc[hcp.monthly_patient_volume == 9999, ['hcp_id', 'specialty', 'monthly_patient_volume']]


monthly_patient_volume summary (raw):
count    2500.000000
mean      300.818800
std       507.165468
min        30.000000
25%       148.000000
50%       234.500000
75%       362.250000
max      9999.000000
Name: monthly_patient_volume, dtype: float64

Rows with the placeholder/entry-error value 9999: 6


,hcp_id,specialty,monthly_patient_volume
226,HCP00227,Pediatrician,9999
273,HCP00274,General Practitioner,9999
916,HCP00917,Pediatrician,9999
947,HCP00948,Pediatrician,9999
964,HCP00965,General Practitioner,9999
2118,HCP02119,Neonatologist,9999


In [9]:
hcp_clean = hcp.copy()
hcp_clean.loc[hcp_clean.monthly_patient_volume == 9999, 'monthly_patient_volume'] = np.nan

# Impute with the specialty median rather than dropping the HCP entirely —
# every other field for these rows is valid and usable.
hcp_clean['monthly_patient_volume'] = hcp_clean['monthly_patient_volume'].fillna(
    hcp_clean.groupby('specialty')['monthly_patient_volume'].transform('median')
)

print("monthly_patient_volume summary (clean):")
print(hcp_clean.monthly_patient_volume.describe())

n_null_digital = hcp_clean.digital_engagement.isna().sum()
print(f"\ndigital_engagement nulls: {n_null_digital} ({n_null_digital/len(hcp_clean):.1%}) — left as null; excluded from digital-specific analyses rather than imputed, since guessing engagement would understate genuine variation")


monthly_patient_volume summary (clean):
count    2500.000000
mean      277.425400
std       175.700384
min        30.000000
25%       148.000000
50%       234.000000
75%       361.000000
max      1556.000000
Name: monthly_patient_volume, dtype: float64

digital_engagement nulls: 50 (2.0%) — left as null; excluded from digital-specific analyses rather than imputed, since guessing engagement would understate genuine variation


## 6. Consumer data — missing price sensitivity

In [10]:
n_null_price = consumer.price_sensitivity.isna().sum()
print(f"price_sensitivity nulls: {n_null_price} ({n_null_price/len(consumer):.1%})")

consumer_clean = consumer.copy()
# Left null rather than imputed: price sensitivity is self-reported and a
# guessed value could distort the price-sensitive segment definition used
# in notebook 03. Downstream segmentation explicitly excludes these rows
# from price-based cuts (see notebook 03).
print("Decision: leave null. Flagged for exclusion from price-based segmentation only.")


price_sensitivity nulls: 45 (1.5%)
Decision: leave null. Flagged for exclusion from price-based segmentation only.


## 7. Reconciliation — does the cleaned sales data match territory_data?

An important check before trusting any territory-level number later: does summing cleaned monthly sales reproduce the `current_sales` figure already computed in `territory_data.csv`?

In [11]:
post_relaunch_by_territory = (
    sales_active[sales_active.market_phase == 'post_relaunch']
    .groupby('territory_id')['revenue'].sum() / 1e5   # -> INR lakh
)

check = territory.set_index('territory_id')['current_sales'].to_frame('territory_file')
check['recomputed_from_sales'] = post_relaunch_by_territory
check['diff_pct'] = 100 * (check.recomputed_from_sales - check.territory_file) / check.territory_file

print(f"Max absolute reconciliation difference: {check.diff_pct.abs().max():.3f}%")
check.sort_values('diff_pct', key=abs, ascending=False).head()


Max absolute reconciliation difference: 0.107%


,territory_file,recomputed_from_sales,diff_pct
territory_id,,,
TR07,1.81,1.808060,-0.107204
TR34,4.28,4.275467,-0.105907
TR29,5.00,5.004749,0.094976
TR11,4.16,4.163454,0.083026
TR25,4.02,4.017286,-0.067520


Reconciles to well under 1%, so the cleaned sales table and the pre-computed territory table are consistent and safe to use interchangeably in later notebooks.

## 8. Save cleaned tables

Written to `../outputs/tables/` so notebooks 02–05 read the cleaned versions rather than repeating this cleaning logic.

In [12]:
import os
os.makedirs('../outputs/tables', exist_ok=True)

sales_clean.to_csv('../outputs/tables/sales_clean.csv', index=False)
sales_active.to_csv('../outputs/tables/sales_active.csv', index=False)
hcp_clean.to_csv('../outputs/tables/hcp_clean.csv', index=False)
consumer_clean.to_csv('../outputs/tables/consumer_clean.csv', index=False)

print("Saved: sales_clean.csv, sales_active.csv, hcp_clean.csv, consumer_clean.csv")


Saved: sales_clean.csv, sales_active.csv, hcp_clean.csv, consumer_clean.csv


## Summary

| Issue found | Rows affected | Action |
|---|---|---|
| Exact duplicate sales rows | 63 | Dropped |
| Inconsistent state labels (4 variants) | ~127 | Mapped to canonical names |
| Off-market zero-sales rows | 4,752 | Kept, tagged `off_market`, excluded from demand averages |
| Recall-returns negative units (Apr 2024) | 144 | Kept (real event), excluded from demand averages |
| Missing `discount_pct` (active rows) | ~78 | Imputed with channel-phase median |
| HCP `monthly_patient_volume` = 9999 (entry error) | 6 | Nulled and imputed with specialty median |
| HCP `digital_engagement` missing | 50 | Left null; excluded from digital-specific cuts |
| Consumer `price_sensitivity` missing | 45 | Left null; excluded from price-based segmentation |

**Reconciliation:** cleaned sales data matches the pre-computed `territory_data.csv` to well under 1%, confirming internal consistency before any further analysis.

**Next:** `02_eda.ipynb` explores the cleaned data for the patterns underlying the case study's core questions.
